<a href="https://colab.research.google.com/github/Shashank-ravi46/Shashank-R/blob/main/Fixed_Income_Analytics_Suite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

!pip install fredapi pandas scipy plotly openpyxl --quiet

print("✅ Done! All tools installed.")


✅ Done! All tools installed.


In [3]:


FRED_API_KEY = "098379be7f580ee7b313c29d37080cfb"
START_DATE   = "2015-01-01"
END_DATE     = "2026-06-19"

print(f"✅ Settings saved. Fetching data from {START_DATE} to {END_DATE}")


✅ Settings saved. Fetching data from 2015-01-01 to 2026-06-19


In [4]:
import pandas as pd
from fredapi import Fred

fred = Fred(api_key=FRED_API_KEY)

# These are the US Treasury yield series on FRED
# Each one is a different maturity (how long until the bond is paid back)
series_to_fetch = {
    "2Y":  "DGS2",    # 2-year Treasury yield
    "5Y":  "DGS5",    # 5-year
    "10Y": "DGS10",   # 10-year (the most famous one)
    "30Y": "DGS30",   # 30-year
}

frames = {}
for name, fred_code in series_to_fetch.items():
    print(f"Downloading {name} yield...")
    frames[name] = fred.get_series(fred_code,
                                   observation_start=START_DATE,
                                   observation_end=END_DATE)

# Combine into one table
yields = pd.DataFrame(frames)
yields.index = pd.to_datetime(yields.index)
yields = yields.dropna()

print(f"\n✅ Data downloaded! Shape: {yields.shape}")
print(f"First date: {yields.index[0].date()}")
print(f"Last date:  {yields.index[-1].date()}")
yields.tail(5)



✅ Data downloaded! Shape: (2866, 4)
First date: 2015-01-02
Last date:  2026-06-17


,2Y,5Y,10Y,30Y
2026-06-11,4.05,4.18,4.45,4.95
2026-06-12,4.09,4.21,4.48,4.97
2026-06-15,4.07,4.18,4.47,4.97
2026-06-16,4.05,4.16,4.43,4.93
2026-06-17,4.20,4.27,4.49,4.93


In [5]:
yields_clean = yields.copy()
yields_clean = yields_clean.ffill(limit=5)   # fill gaps up to 5 days
yields_clean = yields_clean.dropna()          # drop anything still missing

# Add a useful feature: the "slope" of the curve
# If 10Y yield > 2Y yield, the curve is normal (upward sloping)
# If 2Y yield > 10Y yield, the curve is "inverted" — often a recession warning
yields_clean["slope_10y2y"] = yields_clean["10Y"] - yields_clean["2Y"]
yields_clean["inverted"]    = (yields_clean["slope_10y2y"] < 0).astype(int)

inversion_days = yields_clean["inverted"].sum()
print(f"✅ Cleaned! {len(yields_clean)} trading days of data.")
print(f"⚠️  Curve was inverted on {inversion_days} days in this period.")
yields_clean.tail(5)


✅ Cleaned! 2866 trading days of data.
⚠️  Curve was inverted on 544 days in this period.


,2Y,5Y,10Y,30Y,slope_10y2y,inverted
2026-06-11,4.05,4.18,4.45,4.95,0.40,0
2026-06-12,4.09,4.21,4.48,4.97,0.39,0
2026-06-15,4.07,4.18,4.47,4.97,0.40,0
2026-06-16,4.05,4.16,4.43,4.93,0.38,0
2026-06-17,4.20,4.27,4.49,4.93,0.29,0


In [6]:
import numpy as np

def price_bond(face, coupon_rate, maturity_years, ytm, freq=2):
    """
    face         = how much money you get back at the end (e.g. £1000)
    coupon_rate  = annual interest rate the bond pays (e.g. 0.045 = 4.5%)
    maturity_years = how many years until you get your money back
    ytm          = yield to maturity (the market's required return)
    freq         = how many times per year coupons are paid (2 = semi-annual)
    """
    n_periods  = int(maturity_years * freq)           # total number of payments
    coupon     = face * coupon_rate / freq            # payment per period
    rate       = ytm / freq                           # discount rate per period
    periods    = np.arange(1, n_periods + 1)

    # Each coupon payment discounted back to today
    pv_coupons = np.sum(coupon / (1 + rate) ** periods)

    # The face value (principal) at the end, also discounted
    pv_face    = face / (1 + rate) ** n_periods

    return round(pv_coupons + pv_face, 4)


def modified_duration(face, coupon_rate, maturity_years, ytm, freq=2):
    """
    Modified duration tells you: if yields go up by 1%, how much does the price drop?
    A duration of 8 means: yields up 1% → price drops roughly 8%
    """
    price  = price_bond(face, coupon_rate, maturity_years, ytm, freq)
    n      = int(maturity_years * freq)
    c      = face * coupon_rate / freq
    r      = ytm / freq
    t      = np.arange(1, n + 1)
    cf     = np.full(n, c)
    cf[-1] += face

    pv_cf      = cf / (1 + r) ** t
    mac_dur    = np.sum((t / freq) * pv_cf) / price
    mod_dur    = mac_dur / (1 + r)
    dv01       = mod_dur * price * 0.0001             # $ change per 1 basis point

    return round(mac_dur, 4), round(mod_dur, 4), round(dv01, 4)


# --- Test it with a 10-year Treasury bond ---
face       = 1000     # $1,000 par
coupon     = 0.045    # 4.5% annual coupon
maturity   = 10       # 10 years
ytm        = 0.045    # yield = coupon → bond prices at par ($1000)

p          = price_bond(face, coupon, maturity, ytm)
mac, mod, dv01 = modified_duration(face, coupon, maturity, ytm)

print(f"Bond Price:           ${p:,.2f}")
print(f"Macaulay Duration:    {mac} years")
print(f"Modified Duration:    {mod}")
print(f"DV01 (per $1,000):    ${dv01:.4f}")
print()
print("What this means:")
print(f"  If yields rise by 1%, this bond loses roughly {mod:.1f}% of its value.")
print(f"  If yields rise by 1bp (0.01%), you lose ${dv01:.4f} per $1,000 face value.")


Bond Price:           $1,000.00
Macaulay Duration:    8.1614 years
Modified Duration:    7.9819
DV01 (per $1,000):    $0.7982

What this means:
  If yields rise by 1%, this bond loses roughly 8.0% of its value.
  If yields rise by 1bp (0.01%), you lose $0.7982 per $1,000 face value.


In [7]:


import plotly.graph_objects as go

# Use the most recent day of data
latest      = yields_clean.iloc[-1]
maturities  = [2, 5, 10, 30]       # years
yield_vals  = [latest["2Y"], latest["5Y"], latest["10Y"], latest["30Y"]]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=maturities,
    y=yield_vals,
    mode="lines+markers",
    name="Current Yield Curve",
    line=dict(color="#4f98a3", width=3),
    marker=dict(size=10, color="#e8af34"),
))

fig.update_layout(
    title=f"US Treasury Yield Curve — {yields_clean.index[-1].date()}",
    xaxis_title="Maturity (Years)",
    yaxis_title="Yield (%)",
    template="plotly_dark",
    height=450,
)
fig.show()
print("✅ Chart rendered above.")


✅ Chart rendered above.


In [8]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=yields_clean.index,
    y=yields_clean["slope_10y2y"],
    mode="lines",
    name="10Y − 2Y Spread",
    line=dict(color="#4f98a3", width=1.5),
    fill="tozeroy",
    fillcolor="rgba(79,152,163,0.15)",
))

# Red line at zero — below zero = inversion
fig.add_hline(y=0, line_color="#dd6974", line_dash="dash", line_width=2,
              annotation_text="Inversion Line")

fig.update_layout(
    title="10Y − 2Y Treasury Yield Spread (Recession Warning Indicator)",
    xaxis_title="Date",
    yaxis_title="Spread (percentage points)",
    template="plotly_dark",
    height=430,
)
fig.show()
print("✅ Chart rendered above.")


✅ Chart rendered above.


In [9]:
Stress Test — what happens to bond price when yields move?
# Think of it like this: if interest rates shoot up, your bond loses value
# because new bonds pay more than yours
# We test multiple scenarios: small moves, big moves, crashes

import pandas as pd

# Our bond from Cell 5
face     = 1000
coupon   = 0.045
maturity = 10
base_ytm = 0.045

base_price = price_bond(face, coupon, maturity, base_ytm)

# Yield shocks in basis points (100bps = 1%)
shocks_bps = [-200, -100, -50, -25, 0, +25, +50, +100, +200]

rows = []
for shock in shocks_bps:
    new_ytm   = base_ytm + shock / 10000
    if new_ytm <= 0:
        new_ytm = 0.0001
    new_price  = price_bond(face, coupon, maturity, new_ytm)
    dollar_pnl = new_price - base_price
    pct_pnl    = dollar_pnl / base_price * 100

    rows.append({
        "Yield Shock (bps)": shock,
        "New YTM (%)":        round(new_ytm * 100, 3),
        "New Price ($)":      round(new_price, 2),
        "P&L ($)":            round(dollar_pnl, 2),
        "P&L (%)":            round(pct_pnl, 3),
    })

stress_df = pd.DataFrame(rows)
print("✅ Stress Test Results for 10Y Bond ($1,000 face):")
display(stress_df)


Object `move` not found.
✅ Stress Test Results for 10Y Bond ($1,000 face):


,Yield Shock (bps),New YTM (%),New Price ($),P&L ($),P&L (%)
0,-200,2.50,1175.99,175.99,17.599
1,-100,3.50,1083.76,83.76,8.376
2,-50,4.00,1040.88,40.88,4.088
3,-25,4.25,1020.19,20.19,2.019
4,0,4.50,1000.00,0.00,0.000
5,25,4.75,980.28,-19.72,-1.972
6,50,5.00,961.03,-38.97,-3.897
7,100,5.50,923.86,-76.14,-7.614
8,200,6.50,854.61,-145.39,-14.539


In [11]:
# Visualise the stress test
# The curve bends outward — that bend IS convexity
# A straight line would be duration-only. The curve is reality.

fig = go.Figure()

fig.add_trace(go.Bar(
    x=stress_df["Yield Shock (bps)"],
    y=stress_df["P&L ($)"],
    marker_color=["#dd6974" if v < 0 else "#6daa45" for v in stress_df["P&L ($)"]],
    text=[f"${v:+.2f}" for v in stress_df["P&L ($)"]],
    textposition="outside",
    name="P&L ($)",
))

fig.add_hline(y=0, line_color="white", line_dash="dash", line_width=1)

fig.update_layout(
    title="Bond P&L Under Yield Shocks — 10Y Treasury Note",
    xaxis_title="Yield Shock (basis points)",
    yaxis_title="Price Change ($)",
    template="plotly_dark",
    height=450,
)
fig.show()
print("✅ Stress test chart rendered.")

✅ Stress test chart rendered.


In [12]:
Build a 5-bond portfolio and compute total risk


portfolio_specs = [
    {"name": "2Y Note",  "maturity": 2,  "coupon": 0.045, "ytm": 0.050, "face": 1_000_000},
    {"name": "5Y Note",  "maturity": 5,  "coupon": 0.040, "ytm": 0.047, "face": 750_000},
    {"name": "10Y Note", "maturity": 10, "coupon": 0.038, "ytm": 0.046, "face": 500_000},
    {"name": "20Y Bond", "maturity": 20, "coupon": 0.042, "ytm": 0.048, "face": 250_000},
    {"name": "30Y Bond", "maturity": 30, "coupon": 0.044, "ytm": 0.049, "face": 200_000},
]

rows = []
for spec in portfolio_specs:
    price  = price_bond(spec["face"], spec["coupon"], spec["maturity"], spec["ytm"])
    mac, mod, dv01 = modified_duration(spec["face"], spec["coupon"],
                                       spec["maturity"], spec["ytm"])
    rows.append({
        "Bond":              spec["name"],
        "Face Value ($)":    spec["face"],
        "Market Value ($)":  round(price, 2),
        "Coupon":            f"{spec['coupon']*100:.1f}%",
        "YTM":               f"{spec['ytm']*100:.2f}%",
        "Mod Duration":      mod,
        "DV01 ($)":          round(dv01, 2),
    })

portfolio_df = pd.DataFrame(rows)

# Portfolio totals
total_mv    = portfolio_df["Market Value ($)"].sum()
total_dv01  = portfolio_df["DV01 ($)"].sum()
weights     = portfolio_df["Market Value ($)"] / total_mv
wtd_dur     = (weights * portfolio_df["Mod Duration"]).sum()

print("=" * 55)
print("  PORTFOLIO SUMMARY")
print("=" * 55)
print(f"  Total Market Value:    ${total_mv:>12,.0f}")
print(f"  Wtd Avg Mod Duration:  {wtd_dur:>12.3f} years")
print(f"  Total DV01:            ${total_dv01:>12,.2f}")
print(f"\n  Interpretation:")
print(f"  If rates rise 1bp, this portfolio loses ~${total_dv01:,.0f}")
print(f"  If rates rise 100bps, this portfolio loses ~${total_dv01*100:,.0f}")
print("=" * 55)
display(portfolio_df)


  PORTFOLIO SUMMARY
  Total Market Value:    $   2,600,886
  Wtd Avg Mod Duration:         5.734 years
  Total DV01:            $    1,491.39

  Interpretation:
  If rates rise 1bp, this portfolio loses ~$1,491
  If rates rise 100bps, this portfolio loses ~$149,139


,Bond,Face Value ($),Market Value ($),Coupon,YTM,Mod Duration,DV01 ($)
0,2Y Note,1000000,990595.06,4.5%,5.00%,1.8874,186.97
1,5Y Note,750000,726846.73,4.0%,4.70%,4.4680,324.76
2,10Y Note,500000,468224.47,3.8%,4.60%,8.1597,382.06
3,20Y Bond,250000,230851.85,4.2%,4.80%,13.1201,302.88
4,30Y Bond,200000,184368.08,4.4%,4.90%,15.9856,294.72


In [13]:
# CELL 11: Export all your work to a professional Excel file
# This is what you send to recruiters or attach to your GitHub

import openpyxl

filename = "fixed_income_analytics.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    # Sheet 1: Raw yield history
    yields_clean[["2Y","5Y","10Y","30Y","slope_10y2y","inverted"]].to_excel(
        writer, sheet_name="Yield History", index=True)

    # Sheet 2: Stress test results
    stress_df.to_excel(writer, sheet_name="Stress Test", index=False)

    # Sheet 3: Portfolio analytics
    portfolio_df.to_excel(writer, sheet_name="Portfolio", index=False)

    # Sheet 4: Summary stats
    summary = pd.DataFrame([
        ["Total Market Value ($)",    f"${total_mv:,.0f}"],
        ["Wtd Avg Mod Duration",      f"{wtd_dur:.3f} years"],
        ["Total Portfolio DV01 ($)",  f"${total_dv01:,.2f}"],
        ["P&L if +100bps ($)",        f"-${total_dv01*100:,.0f}"],
        ["P&L if -100bps ($)",        f"+${total_dv01*100:,.0f}"],
        ["Data Start",                str(yields_clean.index[0].date())],
        ["Data End",                  str(yields_clean.index[-1].date())],
        ["Total Trading Days",        str(len(yields_clean))],
        ["Inversion Days",            str(yields_clean["inverted"].sum())],
    ], columns=["Metric", "Value"])

    summary.to_excel(writer, sheet_name="Summary", index=False)

print(f"✅ Excel file saved: {filename}")
print("   To download: click the folder icon on the LEFT sidebar → find the file → right-click → Download")


✅ Excel file saved: fixed_income_analytics.xlsx
   To download: click the folder icon on the LEFT sidebar → find the file → right-click → Download


In [14]:
# CELL 12: Save your charts so you can upload them to GitHub
# HTML files are interactive — anyone can open them in a browser

import os
os.makedirs("charts", exist_ok=True)

# Rebuild and save each chart

# Chart 1: Current yield curve
maturities_list = [2, 5, 10, 30]
latest          = yields_clean.iloc[-1]
yield_vals      = [latest["2Y"], latest["5Y"], latest["10Y"], latest["30Y"]]

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=maturities_list, y=yield_vals,
    mode="lines+markers",
    line=dict(color="#4f98a3", width=3),
    marker=dict(size=10, color="#e8af34"),
    name="Yield Curve",
))
fig1.update_layout(
    title=f"US Treasury Yield Curve — {yields_clean.index[-1].date()}",
    xaxis_title="Maturity (Years)",
    yaxis_title="Yield (%)",
    template="plotly_dark", height=450,
)
fig1.write_html("charts/yield_curve.html")

# Chart 2: Slope/inversion history
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=yields_clean.index,
    y=yields_clean["slope_10y2y"],
    mode="lines",
    line=dict(color="#4f98a3", width=1.5),
    fill="tozeroy",
    fillcolor="rgba(79,152,163,0.15)",
    name="10Y−2Y Spread",
))
fig2.add_hline(y=0, line_color="#dd6974", line_dash="dash", line_width=2)
fig2.update_layout(
    title="10Y − 2Y Spread (Recession Indicator)",
    xaxis_title="Date",
    yaxis_title="Spread (pp)",
    template="plotly_dark", height=430,
)
fig2.write_html("charts/slope_history.html")

# Chart 3: Stress test
fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=stress_df["Yield Shock (bps)"],
    y=stress_df["P&L ($)"],
    marker_color=["#dd6974" if v < 0 else "#6daa45" for v in stress_df["P&L ($)"]],
    text=[f"${v:+.2f}" for v in stress_df["P&L ($)"]],
    textposition="outside",
))
fig3.add_hline(y=0, line_color="white", line_dash="dash")
fig3.update_layout(
    title="Bond P&L Under Yield Shocks",
    xaxis_title="Yield Shock (bps)",
    yaxis_title="Price Change ($)",
    template="plotly_dark", height=450,
)
fig3.write_html("charts/stress_test.html")

print("✅ Charts saved:")
print("   charts/yield_curve.html")
print("   charts/slope_history.html")
print("   charts/stress_test.html")
print()
print("Download them the same way: folder icon → charts folder → right-click → Download")


✅ Charts saved:
   charts/yield_curve.html
   charts/slope_history.html
   charts/stress_test.html

Download them the same way: folder icon → charts folder → right-click → Download


In [15]:
# CELL 13: Print your final project summary
# This is what you read out in an interview when they ask "walk me through your project"

print("=" * 60)
print("   FIXED INCOME ANALYTICS SUITE — PROJECT COMPLETE")
print("=" * 60)
print()
print("WHAT YOU BUILT:")
print("  ✅ Downloaded real US Treasury yields from FRED API")
print("  ✅ Cleaned and engineered features (slope, inversion flag)")
print("  ✅ Priced a 10Y bond using present value formula")
print("  ✅ Computed Modified Duration and DV01")
print("  ✅ Ran stress tests across 9 yield shock scenarios")
print("  ✅ Built a 5-bond portfolio with total risk metrics")
print("  ✅ Exported results to Excel (4 sheets)")
print("  ✅ Saved 3 interactive Plotly charts")
print()
print("KEY NUMBERS:")
print(f"  Current 10Y Yield:      {yields_clean['10Y'].iloc[-1]:.3f}%")
print(f"  Current 2Y Yield:       {yields_clean['2Y'].iloc[-1]:.3f}%")
print(f"  10Y−2Y Slope:           {yields_clean['slope_10y2y'].iloc[-1]:.3f}pp")
print(f"  Total Portfolio MV:     ${total_mv:,.0f}")
print(f"  Portfolio DV01:         ${total_dv01:,.2f}")
print(f"  Loss if rates +100bps:  ${total_dv01*100:,.0f}")
print()
print("FILES TO UPLOAD TO GITHUB:")
print("  📄 Your Colab notebook (.ipynb)")
print("  📊 fixed_income_analytics.xlsx")
print("  📈 charts/yield_curve.html")
print("  📈 charts/slope_history.html")
print("  📈 charts/stress_test.html")
print("  📝 README.md  ← we write this next")
print("=" * 60)


   FIXED INCOME ANALYTICS SUITE — PROJECT COMPLETE

WHAT YOU BUILT:
  ✅ Downloaded real US Treasury yields from FRED API
  ✅ Cleaned and engineered features (slope, inversion flag)
  ✅ Priced a 10Y bond using present value formula
  ✅ Computed Modified Duration and DV01
  ✅ Ran stress tests across 9 yield shock scenarios
  ✅ Built a 5-bond portfolio with total risk metrics
  ✅ Exported results to Excel (4 sheets)
  ✅ Saved 3 interactive Plotly charts

KEY NUMBERS:
  Current 10Y Yield:      4.490%
  Current 2Y Yield:       4.200%
  10Y−2Y Slope:           0.290pp
  Total Portfolio MV:     $2,600,886
  Portfolio DV01:         $1,491.39
  Loss if rates +100bps:  $149,139

FILES TO UPLOAD TO GITHUB:
  📄 Your Colab notebook (.ipynb)
  📊 fixed_income_analytics.xlsx
  📈 charts/yield_curve.html
  📈 charts/slope_history.html
  📈 charts/stress_test.html
  📝 README.md  ← we write this next
